<center> <h1><b>PINYASURI FLIGHT METRICS ANALYSIS</b></h1></center>

This notebook computes flight performance metrics from raw telemetry data collected during drone missions.

<h2>Imports & Configuration</h2>

In [13]:
import pandas as pd
import numpy as np
import math

import config

<h2>Load Raw Flight Data</h2>

In [14]:
df = pd.read_csv('raw_flight_data.csv')

# Parse timestamp
df["timestamp"] = pd.to_datetime(df["timestamp_utc"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df)} telemetry rows.")

Loaded 3515 telemetry rows.


<h2>Group by Flight ID:</h2>

In [15]:
grouped = df.groupby("flight_id")

<h2>1. Altitude Stability</h2>

In [17]:
def compute_attitude_stability(df):
    att = df[["roll_deg", "pitch_deg", "yaw_deg"]].dropna()

    return {
        "roll_std_deg":  att["roll_deg"].std(),
        "pitch_std_deg": att["pitch_deg"].std(),
        "yaw_std_deg":   att["yaw_deg"].std(),
    }

In [20]:
attitude_stability = compute_attitude_stability(df)
print("Attitude Stability:", attitude_stability)

Attitude Stability: {'roll_std_deg': 15.34478215651401, 'pitch_std_deg': 11.034631004454003, 'yaw_std_deg': 32.37997347080348}


<h2>2. IMU Vibration (Acceleration RMS)</h2>

In [21]:
def compute_imu_rms(df):
    imu = df[[
        "accel_x_m_s2",
        "accel_y_m_s2",
        "accel_z_m_s2"
    ]].dropna()

    accel_mag = np.sqrt(
        imu["accel_x_m_s2"]**2 +
        imu["accel_y_m_s2"]**2 +
        imu["accel_z_m_s2"]**2
    )

    rms = np.sqrt(np.mean(accel_mag**2))
    return rms

imu_rms = compute_imu_rms(df)
print("IMU Vibration (Acceleration RMS):", imu_rms)

IMU Vibration (Acceleration RMS): 9.831839607259335


<h2>3. Waypoint Navigation Accuracy (Haversine)</h2>

In [22]:
EARTH_RADIUS_M = 6371000  # meters

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2)**2 +
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_M * c

In [23]:
def compute_waypoint_accuracy(df):
    wp = df[
        (df["waypoint_index"] >= 0)
    ].dropna(subset=[
        "lat_deg", "lon_deg",
        "waypoint_lat_deg", "waypoint_lon_deg",
        "alt_m", "waypoint_alt_m"
    ])

    if wp.empty:
        return None

    horizontal_dist = haversine(
        wp["lat_deg"],
        wp["lon_deg"],
        wp["waypoint_lat_deg"],
        wp["waypoint_lon_deg"]
    )

    vertical_dist = np.abs(
        wp["alt_m"] - wp["waypoint_alt_m"]
    )

    total_dist = np.sqrt(horizontal_dist**2 + vertical_dist**2)

    return {
        "mean_horizontal_error_m": horizontal_dist.mean(),
        "rms_horizontal_error_m":  np.sqrt(np.mean(horizontal_dist**2)),
        "mean_3d_error_m":         total_dist.mean(),
        "rms_3d_error_m":          np.sqrt(np.mean(total_dist**2)),
    }

waypoint_accuracy = compute_waypoint_accuracy(df)
print("Waypoint Navigation Accuracy (Haversine):", waypoint_accuracy)

Waypoint Navigation Accuracy (Haversine): {'mean_horizontal_error_m': np.float64(5840380.343611267), 'rms_horizontal_error_m': np.float64(8832156.221523492), 'mean_3d_error_m': np.float64(5840382.9014921775), 'rms_3d_error_m': np.float64(8832156.221525788)}


<h2>4. Position Jitter During Hover</h2>

In [8]:
def compute_hover_jitter(df):
    hover = df[df["is_hovering"] == True].dropna(subset=[
        "lat_deg", "lon_deg", "alt_m"
    ])

    if hover.empty:
        return None

    lat0 = hover["lat_deg"].mean()
    lon0 = hover["lon_deg"].mean()
    alt0 = hover["alt_m"].mean()

    horizontal = haversine(
        hover["lat_deg"],
        hover["lon_deg"],
        lat0,
        lon0
    )

    vertical = np.abs(hover["alt_m"] - alt0)

    pos_error = np.sqrt(horizontal**2 + vertical**2)

    return {
        "hover_rms_jitter_m": np.sqrt(np.mean(pos_error**2)),
        "hover_mean_jitter_m": pos_error.mean(),
        "hover_samples": len(hover)
    }

hover_jitter = compute_hover_jitter(df)
hover_jitter

{'hover_rms_jitter_m': np.float64(4.375363463780026),
 'hover_mean_jitter_m': np.float64(4.024249941167423),
 'hover_samples': 3025}

<h2>5. Flight Endurance</h2>

In [9]:
def compute_flight_endurance(df):
    battery = df.dropna(subset=[
        "timestamp",
        "battery_voltage_V",
        "battery_current_A",
        "battery_percentage"
    ])

    if len(battery) < 2:
        return None

    flight_time_sec = (
        battery["timestamp"].iloc[-1] -
        battery["timestamp"].iloc[0]
    ).total_seconds()

    power_w = (
        battery["battery_voltage_V"] *
        battery["battery_current_A"]
    )

    return {
        "flight_time_min": flight_time_sec / 60,
        "avg_voltage_V": battery["battery_voltage_V"].mean(),
        "avg_current_A": battery["battery_current_A"].mean(),
        "avg_power_W": power_w.mean(),
        "battery_used_pct": (
            battery["battery_percentage"].iloc[0] -
            battery["battery_percentage"].iloc[-1]
        )
    }

flight_endurance = compute_flight_endurance(df)
flight_endurance

{'flight_time_min': 220.48034318333333,
 'avg_voltage_V': np.float64(6.98423613086771),
 'avg_current_A': np.float64(0.02869985775248933),
 'avg_power_W': np.float64(0.35273012802275966),
 'battery_used_pct': np.int64(0)}

<h2>Flight Metrics</h2>

In [10]:
flight_metrics = {
    "attitude_stability": attitude_stability,
    "imu_rms_m_s2": imu_rms,
    "waypoint_accuracy": waypoint_accuracy,
    "hover_jitter": hover_jitter,
    "flight_endurance": flight_endurance,
}

flight_metrics

{'attitude_stability': {'roll_std_deg': 15.34478215651401,
  'pitch_std_deg': 11.034631004454003,
  'yaw_std_deg': 32.37997347080348},
 'imu_rms_m_s2': np.float64(9.831839607259335),
 'waypoint_accuracy': {'mean_horizontal_error_m': np.float64(5840380.343611267),
  'rms_horizontal_error_m': np.float64(8832156.221523492),
  'mean_3d_error_m': np.float64(5840382.9014921775),
  'rms_3d_error_m': np.float64(8832156.221525788)},
 'hover_jitter': {'hover_rms_jitter_m': np.float64(4.375363463780026),
  'hover_mean_jitter_m': np.float64(4.024249941167423),
  'hover_samples': 3025},
 'flight_endurance': {'flight_time_min': 220.48034318333333,
  'avg_voltage_V': np.float64(6.98423613086771),
  'avg_current_A': np.float64(0.02869985775248933),
  'avg_power_W': np.float64(0.35273012802275966),
  'battery_used_pct': np.int64(0)}}

<h2>Create final per-flight summary table:</h2>

In [11]:
summary_df = pd.DataFrame(flight_metrics)
summary_df

,attitude_stability,imu_rms_m_s2,waypoint_accuracy,hover_jitter,flight_endurance
roll_std_deg,15.344782,9.83184,NaN,NaN,NaN
pitch_std_deg,11.034631,9.83184,NaN,NaN,NaN
yaw_std_deg,32.379973,9.83184,NaN,NaN,NaN
mean_horizontal_error_m,NaN,9.83184,5.840380e+06,NaN,NaN
rms_horizontal_error_m,NaN,9.83184,8.832156e+06,NaN,NaN
mean_3d_error_m,NaN,9.83184,5.840383e+06,NaN,NaN
rms_3d_error_m,NaN,9.83184,8.832156e+06,NaN,NaN
hover_rms_jitter_m,NaN,9.83184,NaN,4.375363,NaN
hover_mean_jitter_m,NaN,9.83184,NaN,4.024250,NaN
hover_samples,NaN,9.83184,NaN,3025.000000,NaN


<h2>Save results to CSV:</h2>

In [ ]:
summary_df.to_csv("flight_metrics_summary.csv", index=False)